# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakshmitejaswiniparuchuri/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### ML Task Type: Supervised Ranking & Probability Scoring (via Binary Classification)

My **Content Refresh** lane is best framed as a **risk-scoring and ranking task** (powered by binary classification probabilities) rather than clustering or hard yes/no classification alone.

* **Why Scoring & Ranking Fits Best:** An editorial team has a fixed capacity each sprint (for example, reviewing the top 20 or top 50 pages). Rather than treating every flagged page as equally urgent, predicting a continuous refresh priority score lets us **rank** all eligible pages from highest to lowest urgency so editors work on the highest-leverage candidates first.
* **How It Maps to the ML Loop:** Each page receives an estimated probability / priority score of meaningful traffic decline, which is sorted into a ranked editorial triage queue.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target / Proxy Definition

* **Ideal Ground Truth (What We Actually Care About):** Whether refreshing a given page produces a meaningful, sustained recovery in organic clicks and engagement over the following 30–90 days compared to leaving it untouched.
* **Observed Proxy Target (What We Can Measure in the Starter Dataset):** Since this dataset does not contain randomized refresh intervention outcomes, we define an **observable decline proxy** from the historical performance columns:
  * **Binary Decline Proxy (`needs_refresh_proxy`):** A page is labeled `1` if it has meaningful baseline visibility (e.g., `impressions_90d >= 500` or `impressions_prev_30d >= 100`) **and** shows measurable period-over-period decay (`trend_direction == 'declining'` or `trend_pct <= -0.15`), and `0` otherwise.
  * **Continuous Severity Proxy:** We can also inspect `trend_pct` (percentage change between `prev_30d` and `last_30d`) alongside volume (`impressions_prev_30d`) to weight high-visibility drops more heavily than tiny low-volume fluctuations.
* **Leakage & Caveat Note:** Any features derived directly from the target window (`trend_direction`, `trend_pct`, or `last_30d` deltas used to construct the label) must be **strictly excluded** from the model's input features so the model predicts risk from prior/structural signals (`age_tier`, `days_since_last_update`, `word_count`, `competition`, `avg_position`, `ctr`, `prev_30d` baseline) rather than memorizing the label formula.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metrics: Offline Evaluation & Editorial Action Alignment

Because an editorial team only reviews the top $K$ flagged pages per sprint (e.g., $K = 20$ or $K = 50$), global accuracy is misleading—most pages are not urgent refresh candidates, so a model predicting "no refresh needed" everywhere would look artificially accurate while missing every decaying page.

* **Primary Offline Ranking Metric — Precision@K & Recall@K (e.g., Precision@50):**
  * **Precision@K** measures what fraction of the top $K$ pages surfaced to editors actually experienced meaningful decline. High Precision@K directly protects editorial hours from wasted rewrites (false positives).
  * **Recall@K** measures what share of all truly decaying high-visibility pages we capture within the team's sprint capacity $K$.
* **Secondary Threshold-Free Metric — PR-AUC (Precision-Recall Area Under the Curve):**
  * Evaluates ranking quality across all score thresholds on imbalanced decline labels without assuming a fixed cutoff.
* **Domain Baseline Comparison:**
  * Every ML model must beat a simple non-ML heuristic baseline—such as **sorting purely by `days_since_last_update`** or **sorting by raw `impressions_prev_30d`**—on Precision@K and PR-AUC.
* **Downstream Content Action Tied to the Metric:**
  * The top-$K$ scored pages are exported to a sprint review queue where a content strategist assigns an editor to audit search intent, update stale sections, and improve titles/meta descriptions.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd

# Load the starter dataset (use the exact URL/path from your starter notebook if already present)
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
try:
    df = pd.read_csv(url)
except Exception:
    df = pd.read_csv("../data/raw/content_refresh_anonymized.csv")

# Verify unit of analysis: 1 row = 1 unique page (content_id)
print(f"Total rows: {len(df):,} | Unique content_ids: {df['content_id'].nunique():,}")

# Filter to eligible slice with meaningful prior visibility (e.g., impressions_prev_30d >= 100)
lane_df = df[df["impressions_prev_30d"] >= 100].copy()

# Sketch the binary decline proxy target column
lane_df["target_refresh_proxy"] = (
    (lane_df["trend_direction"] == "declining") & (lane_df["trend_pct"] <= -0.15)
).astype(int)

print(f"Eligible slice (impressions_prev_30d >= 100): {len(lane_df):,} pages")
print("Target proxy class balance (%):")
print((lane_df["target_refresh_proxy"].value_counts(normalize=True) * 100).round(1))

# Display real dataframe slice showing unit of analysis + candidate features + sketched target
cols_to_show = [
    "content_id",
    "client_id",
    "content_type",
    "days_since_last_update",
    "avg_position",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "target_refresh_proxy",
]
display(lane_df[cols_to_show].head(10))

Total rows: 30,000 | Unique content_ids: 30,000
Eligible slice (impressions_prev_30d >= 100): 18,010 pages
Target proxy class balance (%):
target_refresh_proxy
0    100.0
Name: proportion, dtype: float64


,content_id,client_id,content_type,days_since_last_update,avg_position,impressions_prev_30d,impressions_last_30d,trend_pct,target_refresh_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,20,10.6,987,578,-41.4,0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,25,20.3,5915,2501,-57.7,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,20,36.5,6089,2382,-60.9,0
3,content_331d6c4de07b,client_19581e27de,keyword article,22,6.2,4206,3626,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,14,44.0,6452,4211,-34.7,0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,20,8.5,1009,617,-38.9,0
7,content_a63219c6e95a,client_19581e27de,keyword article,22,21.2,632,636,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,20,46.0,13828,5696,-58.8,0
9,content_c27558df2b0c,client_19581e27de,keyword article,104,4.9,356,252,-29.2,0
10,content_d8ee6cc6d642,client_19581e27de,keyword article,104,2.2,6441,7665,19.0,0


* **Unit of Analysis Confirmed:** Each row in `lane_df` represents **one unique page (`content_id`)** observed over rolling 30-day and 90-day windows (`impressions_prev_30d` vs. `impressions_last_30d`), restricted to pages with at least 100 prior-month impressions so low-volume noise does not distort the target proxy.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML Beats a Fixed Rule Here

A simple rule-based heuristic (such as *"refresh every page older than 365 days"* or *"sort by largest raw impression drop"*) breaks down in practice for three reasons:

1. **Age Alone Does Not Mean Decay:** Many evergreen guides remain stable for years without edits, while fast-moving topics lose clicks within months. Filtering by `days_since_last_update` alone wastes editorial hours on healthy evergreen pages (false positives) while missing newer pages that are already slipping.
2. **Non-Linear Feature Interactions:** Whether a page is at risk depends on multiple interacting signals simultaneously—such as `avg_position` (pages sitting at positions 4–12 have far more upside and volatility than position 45), `competition`, `content_type`, `word_count`, and `engagement_rate` / `scroll_rate`. Fixed nested `if/else` rules quickly become brittle across different clients and content types.
3. **Continuous Prioritization Over Hard Cutoffs:** A trained scoring model learns how strongly each combination of structural and baseline signals predicts decline risk, producing a calibrated probability score that ranks the top $K$ candidates to match exact editorial sprint capacity—and can be benchmarked directly against simple rule baselines on Precision@K.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-Check

- [x] **ML task type named:** Defined as priority scoring / ranking (via binary classification probabilities) to support a fixed-capacity editorial queue.
- [x] **Target / proxy defined & leakage noted:** Sketched `target_refresh_proxy` from historical decline signals (`trend_direction` and `trend_pct` on pages with `impressions_prev_30d >= 100`) and explicitly noted that target-window features must be excluded from model inputs.
- [x] **Success metric tied to action:** Selected Precision@K, Recall@K, and PR-AUC against simple rule baselines, directly linked to sprint editorial triage.
- [x] **Real dataframe shown:** Loaded the starter dataset, verified `1 row = 1 unique page (content_id)`, and displayed a 10-row slice with the sketched target column.
- [x] **Why ML beats a rule explained:** Detailed why multi-feature interactions and ranked scoring outperform rigid age or threshold rules.
- [x] **Privacy & careful language:** Used only anonymized IDs (`content_id`, `client_id`) with zero real client names, URLs, or causal claims.